# Synthea Insurance Data — PySpark Ingest into Oracle ADB

Reads Synthea CSVs with **PySpark**, transforms columns, and writes to Oracle ADB via JDBC.
Tables are created first via `oracledb` (DDL is cleaner that way), then PySpark appends data.

### Key PySpark patterns shown
| Pattern | Used for |
|---|---|
| `spark.read.csv()` | Read raw CSV with header |
| `withColumn` + `to_date/to_timestamp` | Parse date/timestamp columns |
| `withColumnRenamed` | Rename a single column |
| `toDF(*names)` | Bulk rename all columns at once |
| `select(*cols)` with `.alias()` | Reorder / rename in one step |
| `.cast()` | Change column type |
| `df.write.jdbc()` | Write to Oracle via JDBC |

## 1. SparkSession — Oracle JDBC driver auto-downloaded from Maven

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [4]:
# ojdbc11 = Oracle JDBC driver; oraclepki = wallet/SSL support for ADB.
# osdt_cert + osdt_core are bundled inside oraclepki 23.x — not on Maven separately.
ORACLE_PKGS = ",".join([
    "com.oracle.database.jdbc:ojdbc11:23.3.0.23.09",
    "com.oracle.database.security:oraclepki:23.3.0.23.09",
])

spark = (
    SparkSession.builder
    .appName("SyntheaIngest")
    .config("spark.jars.packages", ORACLE_PKGS)
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")  # keep small for single-node
    # Oracle ADB is remote — JDBC writes take minutes for large tables.
    # Raise Spark's heartbeat limits so the executor isn't killed mid-write.
    .config("spark.network.timeout", "600s")           # default 120s
    .config("spark.executor.heartbeatInterval", "60s") # default 10s
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/10 23:55:24 WARN Utils: Your hostname, BouroLaptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/10 23:55:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/abourantanis/miniconda3/envs/olist_mcp/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/abourantanis/.ivy2.5.2/cache
The jars for the packages stored in: /home/abourantanis/.ivy2.5.2/jars
com.oracle.database.jdbc#ojdbc11 added as a dependency
com.oracle.database.security#oraclepki added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9027767d-26d7-438b-8cee-57d955a7bdfa;1.0
	conf

Spark 4.1.1 ready


## 2. Credentials (OCI Vault) + JDBC connection string

In [5]:
import oci, json, base64, os, oracledb, traceback, sys
from dotenv import load_dotenv
load_dotenv('/mnt/c/Git_Repos/oci-ai-playground/.env')

True

In [6]:
def _show_tb(shell, etype, evalue, tb, tb_offset=None):
    traceback.print_exception(etype, evalue, tb, file=sys.stdout)
    sys.stdout.flush()
get_ipython().set_custom_exc((Exception,), _show_tb)

config         = oci.config.from_file()
secrets_client = oci.secrets.SecretsClient(config)
bundle         = secrets_client.get_secret_bundle(os.environ['OML_USER_CREDS_SECRET_OCID'])
creds          = json.loads(base64.b64decode(bundle.data.secret_bundle_content.content).decode())

TNS_ADMIN = os.environ['TNS_ADMIN']

# WSL: oracledb does not inherit TNS_ADMIN from the env — must set explicitly
oracledb.defaults.config_dir = TNS_ADMIN

# oracledb — used only for DDL (CREATE / DROP TABLE)
conn = oracledb.connect(user=creds['user_name'], password=creds['password'], dsn=creds['dsn'])
print(f"oracledb connected as: {creds['user_name']}  (v{conn.version})")

# PySpark writes via JDBC — same wallet folder that oracledb uses
JDBC_URL   = f"jdbc:oracle:thin:@{creds['dsn']}?TNS_ADMIN={TNS_ADMIN}"
JDBC_PROPS = {
    "user":     creds['user_name'],
    "password": creds['password'],
    "driver":   "oracle.jdbc.OracleDriver",
    "batchsize": "50000",  # default is 1000 — 50x faster
}
print(f"JDBC URL:   {JDBC_URL[:80]}...")

DATA_DIR = "/mnt/c/Git_Repos/oci-ai-playground/output/csv"

oracledb connected as: OML_USER  (v23.26.2.1.0)
JDBC URL:   jdbc:oracle:thin:@(description= (retry_count=20)(retry_delay=3)(address=(protoco...


## 3. Create tables (DDL via oracledb)

In [5]:
DDL = {
    'syn_patients': '''
        CREATE TABLE syn_patients (
            id VARCHAR2(50) PRIMARY KEY, birthdate DATE, deathdate DATE,
            ssn VARCHAR2(20), drivers VARCHAR2(20), passport VARCHAR2(20),
            prefix VARCHAR2(10), first VARCHAR2(100), middle VARCHAR2(100),
            last VARCHAR2(100), suffix VARCHAR2(10), maiden VARCHAR2(100),
            marital VARCHAR2(5), race VARCHAR2(50), ethnicity VARCHAR2(50),
            gender VARCHAR2(5), birthplace VARCHAR2(200), address VARCHAR2(200),
            city VARCHAR2(100), state VARCHAR2(50), county VARCHAR2(100),
            fips VARCHAR2(20), zip VARCHAR2(10),
            lat NUMBER(10,6), lon NUMBER(10,6),
            healthcare_expenses NUMBER(12,2), healthcare_coverage NUMBER(12,2),
            income NUMBER(10,2)
        )''',
    'syn_payer_transitions': '''
        CREATE TABLE syn_payer_transitions (
            patient VARCHAR2(50), memberid VARCHAR2(50),
            start_date DATE, end_date DATE,
            payer VARCHAR2(50), secondary_payer VARCHAR2(50),
            plan_ownership VARCHAR2(50), owner_name VARCHAR2(200)
        )''',
    'syn_encounters': '''
        CREATE TABLE syn_encounters (
            id VARCHAR2(50) PRIMARY KEY, start_ts TIMESTAMP, stop_ts TIMESTAMP,
            patient VARCHAR2(50), organization VARCHAR2(50), provider VARCHAR2(50),
            payer VARCHAR2(50), encounter_class VARCHAR2(50),
            code VARCHAR2(50), description VARCHAR2(500),
            base_encounter_cost NUMBER(10,2), total_claim_cost NUMBER(10,2),
            payer_coverage NUMBER(10,2),
            reason_code VARCHAR2(50), reason_description VARCHAR2(500)
        )''',
    'syn_conditions': '''
        CREATE TABLE syn_conditions (
            start_date DATE, stop_date DATE,
            patient VARCHAR2(50), encounter VARCHAR2(50),
            system VARCHAR2(100), code VARCHAR2(50), description VARCHAR2(500)
        )''',
    'syn_medications': '''
        CREATE TABLE syn_medications (
            start_ts TIMESTAMP, stop_ts TIMESTAMP,
            patient VARCHAR2(50), payer VARCHAR2(50), encounter VARCHAR2(50),
            code VARCHAR2(50), description VARCHAR2(500),
            base_cost NUMBER(10,2), payer_coverage NUMBER(10,2), dispenses NUMBER,
            total_cost NUMBER(10,2),
            reason_code VARCHAR2(50), reason_description VARCHAR2(500)
        )''',
    'syn_procedures': '''
        CREATE TABLE syn_procedures (
            start_ts TIMESTAMP, stop_ts TIMESTAMP,
            patient VARCHAR2(50), encounter VARCHAR2(50),
            system VARCHAR2(100), code VARCHAR2(50), description VARCHAR2(500),
            base_cost NUMBER(10,2),
            reason_code VARCHAR2(50), reason_description VARCHAR2(500)
        )''',
    'syn_observations': '''
        CREATE TABLE syn_observations (
            date_ts TIMESTAMP, patient VARCHAR2(50), encounter VARCHAR2(50),
            category VARCHAR2(50), code VARCHAR2(50), description VARCHAR2(500),
            value VARCHAR2(500), units VARCHAR2(50), obs_type VARCHAR2(50)
        )''',
    'syn_immunizations': '''
        CREATE TABLE syn_immunizations (
            date_ts TIMESTAMP, patient VARCHAR2(50), encounter VARCHAR2(50),
            code VARCHAR2(50), description VARCHAR2(500), base_cost NUMBER(10,2)
        )''',
    'syn_allergies': '''
        CREATE TABLE syn_allergies (
            start_date DATE, stop_date DATE,
            patient VARCHAR2(50), encounter VARCHAR2(50),
            code VARCHAR2(50), system VARCHAR2(100), description VARCHAR2(500),
            allergy_type VARCHAR2(50), category VARCHAR2(50),
            reaction1_code VARCHAR2(50), reaction1_description VARCHAR2(500), severity1 VARCHAR2(50),
            reaction2_code VARCHAR2(50), reaction2_description VARCHAR2(500), severity2 VARCHAR2(50)
        )''',
    'syn_careplans': '''
        CREATE TABLE syn_careplans (
            id VARCHAR2(50), start_date DATE, stop_date DATE,
            patient VARCHAR2(50), encounter VARCHAR2(50),
            code VARCHAR2(50), description VARCHAR2(500),
            reason_code VARCHAR2(50), reason_description VARCHAR2(500)
        )''',
    'syn_devices': '''
        CREATE TABLE syn_devices (
            start_date DATE, stop_date DATE,
            patient VARCHAR2(50), encounter VARCHAR2(50),
            code VARCHAR2(50), description VARCHAR2(500), udi VARCHAR2(200)
        )''',
    'syn_claims': '''
        CREATE TABLE syn_claims (
            id VARCHAR2(50) PRIMARY KEY, patient_id VARCHAR2(50), provider_id VARCHAR2(50),
            primary_patient_insurance_id VARCHAR2(50), secondary_patient_insurance_id VARCHAR2(50),
            department_id NUMBER, patient_department_id NUMBER,
            diagnosis1 VARCHAR2(50), diagnosis2 VARCHAR2(50), diagnosis3 VARCHAR2(50),
            diagnosis4 VARCHAR2(50), diagnosis5 VARCHAR2(50), diagnosis6 VARCHAR2(50),
            diagnosis7 VARCHAR2(50), diagnosis8 VARCHAR2(50),
            referring_provider_id VARCHAR2(50), appointment_id VARCHAR2(50),
            current_illness_date DATE, service_date DATE,
            supervising_provider_id VARCHAR2(50),
            status1 VARCHAR2(50), status2 VARCHAR2(50), statusp VARCHAR2(50),
            outstanding1 NUMBER(10,2), outstanding2 NUMBER(10,2), outstandingp NUMBER(10,2),
            last_billed_date1 DATE, last_billed_date2 DATE, last_billed_datep DATE,
            healthcare_claim_type_id1 NUMBER, healthcare_claim_type_id2 NUMBER
        )''',
    'syn_claims_transactions': '''
        CREATE TABLE syn_claims_transactions (
            id VARCHAR2(50) PRIMARY KEY, claim_id VARCHAR2(50),
            charge_id NUMBER, patient_id VARCHAR2(50),
            txn_type VARCHAR2(50), amount NUMBER(10,2), method VARCHAR2(50),
            from_date DATE, to_date DATE,
            place_of_service VARCHAR2(50), procedure_code VARCHAR2(50),
            modifier1 VARCHAR2(10), modifier2 VARCHAR2(10),
            diagnosis_ref1 NUMBER, diagnosis_ref2 NUMBER,
            diagnosis_ref3 NUMBER, diagnosis_ref4 NUMBER,
            units NUMBER, department_id NUMBER, notes VARCHAR2(500),
            unit_amount NUMBER(10,2), transfer_out_id NUMBER, transfer_type VARCHAR2(10),
            payments NUMBER(10,2), adjustments NUMBER(10,2),
            transfers NUMBER(10,2), outstanding NUMBER(10,2),
            appointment_id VARCHAR2(50), line_note VARCHAR2(500),
            patient_insurance_id VARCHAR2(50), fee_schedule_id NUMBER,
            provider_id VARCHAR2(50), supervising_provider_id VARCHAR2(50)
        )''',
}

with conn.cursor() as cur:
    for table, ddl in DDL.items():
        try:
            cur.execute(f'DROP TABLE {table} PURGE')
            print(f'  dropped  {table}')
        except oracledb.DatabaseError as e:
            if 'ORA-00942' not in str(e):  # anything other than "table does not exist"
                print(f'  DROP {table}: {e}')
        try:
            cur.execute(ddl)
            print(f'  created  {table}')
        except oracledb.DatabaseError as e:
            if 'ORA-00955' in str(e):      # already exists — DROP must have failed silently
                print(f'  {table} already exists, skipping')
            else:
                raise
conn.commit()
print('All tables ready.')

  dropped  syn_patients
  created  syn_patients
  dropped  syn_payer_transitions
  created  syn_payer_transitions
  dropped  syn_encounters
  created  syn_encounters
  dropped  syn_conditions
  created  syn_conditions
  dropped  syn_medications
  created  syn_medications
  dropped  syn_procedures
  created  syn_procedures
  dropped  syn_observations
  created  syn_observations
  dropped  syn_immunizations
  created  syn_immunizations
  dropped  syn_allergies
  created  syn_allergies
  dropped  syn_careplans
  created  syn_careplans
  dropped  syn_devices
  created  syn_devices
  dropped  syn_claims
  created  syn_claims
  dropped  syn_claims_transactions
  created  syn_claims_transactions
All tables ready.


## 4. PySpark helpers

`coalesce(n)` before writing merges partitions — prevents Spark opening 200 concurrent Oracle connections.
For the 2 GB `claims_transactions`, Spark reads it lazily across tasks — no manual chunking needed.

In [6]:
TS_FMT   = "yyyy-MM-dd'T'HH:mm:ss'Z'"  # Synthea timestamp: 1996-11-02T02:35:55Z
DATE_FMT = "yyyy-MM-dd"

def read_csv(name):
    """Read a Synthea CSV — all columns as strings, header=true."""
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")  # read as string, we cast explicitly below
        .option("nullValue", "")          # empty string -> null
        .csv(f"{DATA_DIR}/{name}.csv")
    )

def to_oracle(df, table, partitions=4):
    """Write a Spark DataFrame to Oracle via JDBC."""
    df.coalesce(partitions).write.jdbc(
        url=JDBC_URL, table=table, mode="append", properties=JDBC_PROPS
    )
    print(f"  {table:<35} written")

print('Helpers ready.')

Helpers ready.


## 5. Load tables

In [7]:
# ── PATIENTS ──────────────────────────────────────────────────────────────────
# Pattern: withColumn to cast types, then toDF() to bulk-rename all columns

df = read_csv('patients')

df = (
    df
    .withColumn('BIRTHDATE',           F.to_date('BIRTHDATE', DATE_FMT))
    .withColumn('DEATHDATE',           F.to_date('DEATHDATE', DATE_FMT))
    .withColumn('LAT',                 F.col('LAT').cast('double'))
    .withColumn('LON',                 F.col('LON').cast('double'))
    .withColumn('HEALTHCARE_EXPENSES', F.col('HEALTHCARE_EXPENSES').cast('double'))
    .withColumn('HEALTHCARE_COVERAGE', F.col('HEALTHCARE_COVERAGE').cast('double'))
    .withColumn('INCOME',              F.col('INCOME').cast('double'))
    .toDF('id','birthdate','deathdate','ssn','drivers','passport','prefix',
          'first','middle','last','suffix','maiden','marital','race','ethnicity',
          'gender','birthplace','address','city','state','county','fips','zip',
          'lat','lon','healthcare_expenses','healthcare_coverage','income')
)

print(f'Patients: {df.count():,} rows')
df.printSchema()
to_oracle(df, 'syn_patients')

Patients: 22,920 rows
root
 |-- id: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- deathdate: date (nullable = true)
 |-- ssn: string (nullable = true)
 |-- drivers: string (nullable = true)
 |-- passport: string (nullable = true)
 |-- prefix: string (nullable = true)
 |-- first: string (nullable = true)
 |-- middle: string (nullable = true)
 |-- last: string (nullable = true)
 |-- suffix: string (nullable = true)
 |-- maiden: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthplace: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- fips: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- healthcare_expenses: double (n

26/05/03 21:32:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/03 21:32:49 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:32:49 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


  syn_patients                        written


In [8]:
# ── PAYER_TRANSITIONS ─────────────────────────────────────────────────────────
# Pattern: withColumnRenamed for individual column renames
# START_DATE/END_DATE are full timestamps in the CSV (e.g. 1969-01-19T14:49:29Z)
# — parse as timestamp first, then cast to date.

df = read_csv('payer_transitions')

df = (
    df
    .withColumn('START_DATE', F.to_timestamp('START_DATE', TS_FMT).cast('date'))
    .withColumn('END_DATE',   F.to_timestamp('END_DATE',   TS_FMT).cast('date'))
    .withColumnRenamed('PATIENT',         'patient')
    .withColumnRenamed('MEMBERID',        'memberid')
    .withColumnRenamed('START_DATE',      'start_date')
    .withColumnRenamed('END_DATE',        'end_date')
    .withColumnRenamed('PAYER',           'payer')
    .withColumnRenamed('SECONDARY_PAYER', 'secondary_payer')
    .withColumnRenamed('PLAN_OWNERSHIP',  'plan_ownership')
    .withColumnRenamed('OWNER_NAME',      'owner_name')
)

print(f'Payer transitions: {df.count():,} rows')
# partitions=2 — fewer concurrent connections; each commits faster → avoids heartbeat timeout
to_oracle(df, 'syn_payer_transitions', partitions=2)

Payer transitions: 848,238 rows


26/05/03 21:32:59 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:32:59 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


  syn_payer_transitions               written


In [9]:
# ── ENCOUNTERS ────────────────────────────────────────────────────────────────
# Pattern: select() with F.col().alias() — rename + reorder in one step

df = read_csv('encounters')

df = (
    df
    .withColumn('START',              F.to_timestamp('START', TS_FMT))
    .withColumn('STOP',               F.to_timestamp('STOP',  TS_FMT))
    .withColumn('BASE_ENCOUNTER_COST', F.col('BASE_ENCOUNTER_COST').cast('double'))
    .withColumn('TOTAL_CLAIM_COST',    F.col('TOTAL_CLAIM_COST').cast('double'))
    .withColumn('PAYER_COVERAGE',      F.col('PAYER_COVERAGE').cast('double'))
    .select(
        F.col('Id').alias('id'),
        F.col('START').alias('start_ts'),
        F.col('STOP').alias('stop_ts'),
        F.col('PATIENT').alias('patient'),
        F.col('ORGANIZATION').alias('organization'),
        F.col('PROVIDER').alias('provider'),
        F.col('PAYER').alias('payer'),
        F.col('ENCOUNTERCLASS').alias('encounter_class'),
        F.col('CODE').alias('code'),
        F.col('DESCRIPTION').alias('description'),
        F.col('BASE_ENCOUNTER_COST').alias('base_encounter_cost'),
        F.col('TOTAL_CLAIM_COST').alias('total_claim_cost'),
        F.col('PAYER_COVERAGE').alias('payer_coverage'),
        F.col('REASONCODE').alias('reason_code'),
        F.col('REASONDESCRIPTION').alias('reason_description'),
    )
)

print(f'Encounters: {df.count():,} rows')
to_oracle(df, 'syn_encounters')

Encounters: 1,314,807 rows


26/05/03 21:35:10 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:35:10 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:35:10 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:35:10 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


  syn_encounters                      written


In [10]:
# ── CONDITIONS ────────────────────────────────────────────────────────────────

df = (
    read_csv('conditions')
    .withColumn('START', F.to_date('START', DATE_FMT))
    .withColumn('STOP',  F.to_date('STOP',  DATE_FMT))
    .toDF('start_date','stop_date','patient','encounter','system','code','description')
)

print(f'Conditions: {df.count():,} rows')
to_oracle(df, 'syn_conditions')

Conditions: 820,605 rows


26/05/03 21:41:36 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:41:36 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:41:36 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:41:36 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


  syn_conditions                      written


In [11]:
# ── MEDICATIONS ───────────────────────────────────────────────────────────────

df = (
    read_csv('medications')
    .withColumn('START',          F.to_timestamp('START', TS_FMT))
    .withColumn('STOP',           F.to_timestamp('STOP',  TS_FMT))
    .withColumn('BASE_COST',      F.col('BASE_COST').cast('double'))
    .withColumn('PAYER_COVERAGE', F.col('PAYER_COVERAGE').cast('double'))
    .withColumn('DISPENSES',      F.col('DISPENSES').cast('int'))
    .withColumn('TOTALCOST',      F.col('TOTALCOST').cast('double'))
    .toDF('start_ts','stop_ts','patient','payer','encounter','code','description',
          'base_cost','payer_coverage','dispenses','total_cost','reason_code','reason_description')
)

print(f'Medications: {df.count():,} rows')
to_oracle(df, 'syn_medications')

Medications: 1,102,706 rows


26/05/03 21:43:54 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:43:54 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:43:54 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 21:43:54 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


  syn_medications                     written


In [12]:
# ── PROCEDURES ────────────────────────────────────────────────────────────────

df = (
    read_csv('procedures')
    .withColumn('START',     F.to_timestamp('START', TS_FMT))
    .withColumn('STOP',      F.to_timestamp('STOP',  TS_FMT))
    .withColumn('BASE_COST', F.col('BASE_COST').cast('double'))
    .toDF('start_ts','stop_ts','patient','encounter','system','code','description',
          'base_cost','reason_code','reason_description')
)

print(f'Procedures: {df.count():,} rows')
to_oracle(df, 'syn_procedures')

Procedures: 3,649,054 rows


26/05/03 22:08:34 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:08:34 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:08:34 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:08:34 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


  syn_procedures                      written


In [ ]:
# ── OBSERVATIONS (621 MB) ─────────────────────────────────────────────────────
# PySpark reads this lazily — no need to chunk like pandas did.
# partitions=8 → 8 parallel JDBC connections writing to Oracle.

df = (
    read_csv('observations')
    .withColumn('DATE', F.to_timestamp('DATE', TS_FMT))
    .toDF('date_ts','patient','encounter','category','code','description',
          'value','units','obs_type')
)

print(f'Observations: {df.count():,} rows')
to_oracle(df, 'syn_observations', partitions=8)

Observations: 16,937,728 rows


26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2
26/05/03 22:18:52 WARN JdbcUtils: Requested isolation level 1 is not supported; falling back to default isolation level 2


In [ ]:
# ── IMMUNIZATIONS ─────────────────────────────────────────────────────────────

df = (
    read_csv('immunizations')
    .withColumn('DATE',      F.to_timestamp('DATE', TS_FMT))
    .withColumn('BASE_COST', F.col('BASE_COST').cast('double'))
    .toDF('date_ts','patient','encounter','code','description','base_cost')
)

print(f'Immunizations: {df.count():,} rows')
to_oracle(df, 'syn_immunizations')

In [ ]:
# ── ALLERGIES ─────────────────────────────────────────────────────────────────

df = (
    read_csv('allergies')
    .withColumn('START', F.to_date('START', DATE_FMT))
    .withColumn('STOP',  F.to_date('STOP',  DATE_FMT))
    .toDF('start_date','stop_date','patient','encounter','code','system','description',
          'allergy_type','category',
          'reaction1_code','reaction1_description','severity1',
          'reaction2_code','reaction2_description','severity2')
)

print(f'Allergies: {df.count():,} rows')
to_oracle(df, 'syn_allergies')

In [ ]:
# ── CAREPLANS ─────────────────────────────────────────────────────────────────

df = (
    read_csv('careplans')
    .withColumn('START', F.to_date('START', DATE_FMT))
    .withColumn('STOP',  F.to_date('STOP',  DATE_FMT))
    .toDF('id','start_date','stop_date','patient','encounter','code','description',
          'reason_code','reason_description')
)

print(f'Careplans: {df.count():,} rows')
to_oracle(df, 'syn_careplans')

In [ ]:
# ── DEVICES ───────────────────────────────────────────────────────────────────
# START/STOP are ISO timestamps in the CSV (e.g. 1954-11-14T04:49:29Z),
# not plain dates — parse as timestamp first, then cast to date.

df = (
    read_csv('devices')
    .withColumn('START', F.to_timestamp('START', TS_FMT).cast('date'))
    .withColumn('STOP',  F.to_timestamp('STOP',  TS_FMT).cast('date'))
    .toDF('start_date','stop_date','patient','encounter','code','description','udi')
)

print(f'Devices: {df.count():,} rows')
to_oracle(df, 'syn_devices')

In [ ]:
# ── CLAIMS ────────────────────────────────────────────────────────────────────
# All "date" columns in claims.csv are actually ISO timestamps (e.g. 2020-09-06T03:17:51Z).
# Truncate first — a previous partial write may have left rows that violate the PK.
with conn.cursor() as cur:
    cur.execute('TRUNCATE TABLE syn_claims')
conn.commit()
print('syn_claims truncated')

df = read_csv('claims')

for c in ['CURRENTILLNESSDATE','SERVICEDATE','LASTBILLEDDATE1','LASTBILLEDDATE2','LASTBILLEDDATEP']:
    df = df.withColumn(c, F.to_timestamp(c, TS_FMT).cast('date'))
for c in ['DEPARTMENTID','PATIENTDEPARTMENTID','OUTSTANDING1','OUTSTANDING2','OUTSTANDINGP',
          'HEALTHCARECLAIMTYPEID1','HEALTHCARECLAIMTYPEID2']:
    df = df.withColumn(c, F.col(c).cast('double'))

df = df.toDF(
    'id','patient_id','provider_id',
    'primary_patient_insurance_id','secondary_patient_insurance_id',
    'department_id','patient_department_id',
    'diagnosis1','diagnosis2','diagnosis3','diagnosis4',
    'diagnosis5','diagnosis6','diagnosis7','diagnosis8',
    'referring_provider_id','appointment_id',
    'current_illness_date','service_date','supervising_provider_id',
    'status1','status2','statusp',
    'outstanding1','outstanding2','outstandingp',
    'last_billed_date1','last_billed_date2','last_billed_datep',
    'healthcare_claim_type_id1','healthcare_claim_type_id2'
)

print(f'Claims: {df.count():,} rows')
to_oracle(df, 'syn_claims', partitions=2)

In [ ]:
# ── CLAIMS_TRANSACTIONS (2 GB) ────────────────────────────────────────────────
# Spark splits the file across tasks automatically — no manual chunking needed.
# FROMDATE/TODATE are ISO timestamps in the CSV, not plain dates.

df = read_csv('claims_transactions')

for c in ['FROMDATE','TODATE']:
    df = df.withColumn(c, F.to_timestamp(c, TS_FMT).cast('date'))
for c in ['CHARGEID','AMOUNT','DIAGNOSISREF1','DIAGNOSISREF2','DIAGNOSISREF3','DIAGNOSISREF4',
          'UNITS','DEPARTMENTID','UNITAMOUNT','TRANSFEROUTID',
          'PAYMENTS','ADJUSTMENTS','TRANSFERS','OUTSTANDING','FEESCHEDULEID']:
    df = df.withColumn(c, F.col(c).cast('double'))

df = df.toDF(
    'id','claim_id','charge_id','patient_id','txn_type','amount','method',
    'from_date','to_date','place_of_service','procedure_code',
    'modifier1','modifier2',
    'diagnosis_ref1','diagnosis_ref2','diagnosis_ref3','diagnosis_ref4',
    'units','department_id','notes','unit_amount','transfer_out_id','transfer_type',
    'payments','adjustments','transfers','outstanding',
    'appointment_id','line_note','patient_insurance_id','fee_schedule_id',
    'provider_id','supervising_provider_id'
)

print(f'Claims transactions: {df.count():,} rows')
# partitions=2 — fewer concurrent connections; each commits a large sequential batch
# → avoids heartbeat timeout that killed the executor at partitions=8
to_oracle(df, 'syn_claims_transactions', partitions=2)

## 6. Verify row counts

In [7]:
tables = [
    'syn_patients','syn_payer_transitions','syn_encounters',
    'syn_conditions','syn_medications','syn_procedures',
    'syn_observations','syn_immunizations','syn_allergies',
    'syn_careplans','syn_devices','syn_claims','syn_claims_transactions'
]

total = 0
with conn.cursor() as cur:
    for t in tables:
        cur.execute(f'SELECT COUNT(*) FROM {t}')
        n = cur.fetchone()[0]
        total += n
        print(f'  {t:<35} {n:>10,}')
print(f'  {"TOTAL":<35} {total:>10,}')

  syn_patients                            22,920
  syn_payer_transitions                  848,238
  syn_encounters                       1,314,807
  syn_conditions                         820,605
  syn_medications                      1,102,706
  syn_procedures                       3,649,054
  syn_observations                    16,937,728
  syn_immunizations                            0
  syn_allergies                                0
  syn_careplans                                0
  syn_devices                                  0
  syn_claims                           2,417,513
  syn_claims_transactions             21,345,339
  TOTAL                               48,458,910


## 7. Quick PySpark exploration

Read back from Oracle into Spark and run some analytics — no SQL needed.

In [ ]:
conditions_df = spark.read.jdbc(url=JDBC_URL, table='syn_conditions', properties=JDBC_PROPS)
patients_df   = spark.read.jdbc(url=JDBC_URL, table='syn_patients',   properties=JDBC_PROPS)

print('Top 10 conditions by frequency:')
(
    conditions_df
    .groupBy('description')
    .count()
    .orderBy(F.desc('count'))
    .show(10, truncate=False)
)

print('Patient gender split:')
patients_df.groupBy('gender').count().show()

print('Race/ethnicity breakdown:')
patients_df.groupBy('race','ethnicity').count().orderBy(F.desc('count')).show(10)